<a href="https://colab.research.google.com/github/dkamalakar/rag-demo/blob/feat%2Fdocument_loaders/Document_retriever_search_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies - Langchain etc



In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-huggingface
!pip install jq
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.52
    Uninstalling langchain-core-0.3.52:
      Successfully uninstalled langchain-core-0.3.52
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.23
    Uninstalling langchain-0.3.23:
      Successfully uninstalled langchain-0.3.23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [2]:
!pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.0 MB/s eta 0:00:0

In [3]:
from getpass import getpass

OPENAI_KEY = getpass("Enter your OpenAI Key")

Enter your OpenAI Key··········


In [4]:
import os
os.environ['OPENAI_API_KEY'] = OPENAI_KEY

In [5]:
from langchain_openai import OpenAIEmbeddings
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')


In [6]:
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 15.5MB/s]


In [7]:
!unzip rag_docs.zip

Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  


In [9]:
from langchain.document_loaders import JSONLoader
loader = JSONLoader(
    file_path='./rag_docs/wikidata_rag_demo.jsonl',
    jq_schema='.',
    text_content=False,
    json_lines=True
)

wiki_docs = loader.load()

In [16]:
len(wiki_docs)

1801

In [17]:
wiki_docs[500]

Document(metadata={'source': '/content/rag_docs/wikidata_rag_demo.jsonl', 'seq_num': 501}, page_content='{"id": "99224", "title": "Christopher Lee", "paragraphs": ["Sir Christopher Frank Carandini Lee CBE, CStJ (27 May 1922 7 June 2015) was an English actor. He was a direct descendent of Marie Carandini and Robert E. Lee.", "Lee was best known for his many movie characters. For example, Dracula and Fu Manchu in many movies during the 1950s through the 1970s. He became even better known as Scaramanga in the Bond movie \\"The Man with the Golden Gun\\", Saruman the White in \\"The Lord of the Rings\\" trilogy and Count Dooku in \\"\\".", "He performed roles in 275 movies since 1946. He appeared in over 75 television programs. Lee was also a voice actor, providing his voice in over 50 movies. This made him the Guinness World Record holder for most movie acting roles ever. Lee was also a singer and his latest album was released in 2013."]}')

In [18]:
import json
from langchain.docstore.document import Document
wiki_docs_processed = []

for doc in wiki_docs:
  doc = json.loads(doc.page_content)
  medata = {
      "title": doc['title'],
      "id": doc['id'],
      "source": 'Wikipedia'
  }

  data = ' '.join(doc['paragraphs'])
  wiki_docs_processed.append(Document(page_content=data,medata=medata))

In [19]:
wiki_docs_processed[1500]

Document(metadata={}, page_content='Jan Persson ("Janne Lucas"), born 3 October 1947 in Gothenburg\'s Gamlestad Parish in Gothenburg, Sweden is a Swedish pianist and singer, scoring several chart successes in Sweden during the 1970s and 1980s. Janne Lucas participated at Melodifestivalen 1980 with the song "Växeln hallå", winning the contest. The upcoming year he participated with the song "Rocky Mountain" ending up third. For many years, Janne Lucas also acted as pianist for "Vi i femman" Janne also accompanied the vocal group "Noviserna" for a while, where Anna-Lisa Cederquist participated.')

In [21]:
from langchain.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader("./rag_docs/attention_paper.pdf")
doc_pages = loader.load()

In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=3500,
                                          chunk_overlap=0,


)

doc_chunks = splitter.split_documents(doc_pages)

In [24]:
len(doc_chunks)

16

In [25]:
big_doc = '\n'.join([doc.page_content for doc in doc_chunks])

In [26]:
len(big_doc.split(' '))

5050

In [27]:
from langchain_openai import ChatOpenAI
chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [37]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

def generate_chunk_context(document, chunk):

  chunk_process_prompt = """You are an AI assistant specializing in research paper analysis.
                        Your task is to provide brief, relevant context for a chunk of text based
                        on the following research paper.

                        Here is the research paper:
                        <paper>
                        {paper}
                        </paper>

                        Here is the chunk we want to situate within the whole document.
                        <chunk>
                        {chunk}
                        </chunk>


                        provide a concise context to situate this chunk within the overall document for the
                        purposes of improving search retrieval of the chunk.
                        - Answer only with the succint context and nothing else.
                        - Context should be mentioned like 'Focuses on.....'
                        do not mention 'this chunk or section focuses on...'

                        context:

                        """

  prompt_template = ChatPromptTemplate.from_template(chunk_process_prompt)

  agentic_chunk_chain = (prompt_template
                              |
                         chatgpt
                              |
                         StrOutputParser()


                         )

  context = agentic_chunk_chain.invoke({'paper': document, 'chunk': chunk})

  return context

In [38]:
print(doc_chunks[5].page_content)

output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O
where headi = Attention(QW Q
i , KW K
i , V W V
i )
Where the projections are parameter matrices W Q
i
∈Rdmodel×dk, W K
i
∈Rdmodel×dk, W V
i
∈Rdmodel×dv
and W O ∈Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3
Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and

In [39]:
generate_chunk_context(big_doc, doc_chunks[5].page_content)


'Focuses on the implementation and functionality of multi-head attention in the Transformer model, detailing how it allows for parallel processing of information from different representation subspaces and its application in both encoder-decoder and self-attention layers.'

In [41]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def create_contextual_chunks(file_path):
  print('Loading pages:', file_path)
  loader = PyMuPDFLoader(file_path)
  doc_pages = loader.load()

  print("Chunk pages:", file_path)
  splitter = RecursiveCharacterTextSplitter(chunk_size=3500, chunk_overlap=0)
  doc_chunks = splitter.split_documents(doc_pages)

  print("Generating contextual chunks: ", file_path)

  original_doc = '\n'.join([doc.page_content for doc in doc_chunks])
  contextual_chunks = []

  for chunk in doc_chunks:
    context = generate_chunk_context(original_doc, chunk.page_content)
    contextual_chunks.append(Document(page_content=context+'\n'+chunk.page_content,
                                      metadata=chunk.metadata))

  print("Finished Processing : ", file_path)
  print()
  return contextual_chunks



In [42]:
from glob import glob
pdf_files = glob('./rag_docs/*.pdf')
pdf_files



['./rag_docs/resnet_paper.pdf',
 './rag_docs/cnn_paper.pdf',
 './rag_docs/attention_paper.pdf',
 './rag_docs/vision_transformer.pdf']

In [43]:
paper_docs = []
for fp in pdf_files:
  paper_docs.extend(create_contextual_chunks(fp))

Loading pages: ./rag_docs/resnet_paper.pdf
Chunk pages: ./rag_docs/resnet_paper.pdf
Generating contextual chunks:  ./rag_docs/resnet_paper.pdf
Finished Processing :  ./rag_docs/resnet_paper.pdf

Loading pages: ./rag_docs/cnn_paper.pdf
Chunk pages: ./rag_docs/cnn_paper.pdf
Generating contextual chunks:  ./rag_docs/cnn_paper.pdf
Finished Processing :  ./rag_docs/cnn_paper.pdf

Loading pages: ./rag_docs/attention_paper.pdf
Chunk pages: ./rag_docs/attention_paper.pdf
Generating contextual chunks:  ./rag_docs/attention_paper.pdf
Finished Processing :  ./rag_docs/attention_paper.pdf

Loading pages: ./rag_docs/vision_transformer.pdf
Chunk pages: ./rag_docs/vision_transformer.pdf
Generating contextual chunks:  ./rag_docs/vision_transformer.pdf
Finished Processing :  ./rag_docs/vision_transformer.pdf



In [44]:
len(paper_docs)

79

In [45]:
paper_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'source': './rag_docs/resnet_paper.pdf', 'file_path': './rag_docs/resnet_paper.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'trapped': '', 'modDate': 'D:20151211011345Z', 'creationDate': 'D:20151211011345Z', 'page': 0}, page_content='Focuses on the introduction of a residual learning framework to facilitate the training of deeper neural networks, addressing challenges such as vanishing gradients and degradation of accuracy with increased depth. It highlights the empirical success of residual networks on the ImageNet dataset and their application in various visual recognition tasks, including significant improvements in object detection.\nDeep Residual Learning for Image Recognition\nKaiming He\nXiangyu Zhang\nShaoqing Ren\nJian Sun\nMicrosoft Research\n{k

In [46]:
len(wiki_docs_processed)
total_docs = wiki_docs_processed + paper_docs
len(total_docs)

1880

In [48]:
from langchain_chroma import Chroma

chroma_db = Chroma.from_documents(documents=total_docs,
                                  collection_name='my_db',
                                  embedding=openai_embed_model,
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./my_db")

In [49]:
#Load from disk
chroma_db = Chroma(persist_directory="./my_db",
                   collection_name='my_db',
                   embedding_function=openai_embed_model)

In [50]:
chroma_db